In [1]:
import os
import boto3
from sagemaker import get_execution_role
import time
from pprint import pprint
import shutil

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# function name
str_function_name = 'genxii-pd-tuning-2'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

boto3==1.24.59
pandas==1.2.4
catboost==1.0.4
scikit_learn==0.24.1

Writing requirements.txt


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import os
import pandas as pd
import numpy as np
import catboost as cb
import sklearn.metrics as skm
import boto3
import pickle
import json

# change wd to absolute path
os.chdir('/tmp')

# get metric
def get_metric_by_year_month(target, y_hat, str_eval_metric):
    if str_eval_metric == 'AUC':
        return skm.roc_auc_score(y_true=target, y_score=y_hat)
    elif str_eval_metric == 'PRAUC':
        return skm.average_precision_score(y_true=target, y_score=y_hat)
    elif str_eval_metric == 'Logloss':
        return skm.log_loss(y_true=target, y_pred=y_hat)
    elif str_eval_metric == 'F1':
        return skm.f1_score(y_true=target, y_pred=y_hat)
    else:
        pass

# upload to s3
def upload_to_s3(str_local_path, str_bucket_path, str_project):
    boto3.resource('s3').Bucket(str_project).Object(str_bucket_path).upload_file(str_local_path)

# lambda handler
def lambda_handler(event, context):
    # get index of array
    int_idx_array = int(event['int_idx'])
    print(f'Array index: {int_idx_array}')

    # constants
    str_project = '20231010-gen-xii'
    str_target = 'target'
    #str_dirname_output = './output'
    str_dirname_output = '.'

#     # create output dir
#     try:
#         os.mkdir(str_dirname_output)
#     except FileExistsError:
#         pass

    ###############################################################################
    # HYPERPARAMETERS
    ###############################################################################
    # get df_hyperparameters
    str_filename = 'df_hyperparameters.csv'
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/02_model/12_step_function/{str_filename}'
    df = pd.read_csv(str_uri)
    # convert to dict
    dict_hyperparameters = dict(zip(df['keys'], df['values']))

    # get number of iterations
    int_n_iterations = int(dict_hyperparameters['INT_N_ITERATIONS'])
    print(f'Iterations: {int_n_iterations}')

    # get filename for training
    str_filename_train = dict_hyperparameters['STR_FILENAME_TRAIN']
    print(f'Training filename: {str_filename_train}')

    # get filename for valid
    str_filename_valid = dict_hyperparameters['STR_FILENAME_VALID']
    print(f'Valid filename: {str_filename_valid}')

    # get proportion of iterations to use as early stopping rounds
    flt_prop_early_stopping = float(dict_hyperparameters['PROP_EARLY_STOPPING'])
    print(f'Proportion early stopping: {flt_prop_early_stopping}')

    # get eval metric
    str_eval_metric = dict_hyperparameters['STR_EVAL_METRIC']
    print(f'Eval metric: {str_eval_metric}')

    ##################################################################################

    # get list of features
    print('Importing list_of_starting_features...')
    str_filename = 'df_cols_in_model.csv'
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/02_model/01_lambda_get_starting_feats/{str_filename}'
    list_cols_model = list(pd.read_csv(str_uri)['feature'])
    # add target
    list_cols_import = list_cols_model + [str_target]

    # get lr
    print('Getting learning rate...')
    int_n_tuning_jobs_feat_select = 100 # could include this in hyperparameters
    flt_learning_rate = list(np.around(np.linspace(0.001, 0.999, int_n_tuning_jobs_feat_select), 4))[int_idx_array] # round each value to 4 decimal places

    # read training data
    print('Reading training data...')
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/00_preprocessing/02_make_dfs/{str_filename_train}'
    df = pd.read_parquet(str_uri, columns=list_cols_import)

    # get the non numeric feats
    print('Getting list of non-numeric columns...')
    list_cols_non_numeric = []
    for col in list_cols_model:
        if df[col].dtype not in ['int64','float64']:
            list_cols_non_numeric.append(col)

    # build model
    print('Building model...')
    # pool data
    pool_train = cb.Pool(
        df[list_cols_model], 
        df[str_target], 
        cat_features=list_cols_non_numeric,
    )
    del df

    # read validation data
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/00_preprocessing/02_make_dfs/{str_filename_valid}'
    df = pd.read_parquet(str_uri, columns=list_cols_import)

    # pool data
    pool_valid = cb.Pool(
        df[list_cols_model], 
        df[str_target], 
        cat_features=list_cols_non_numeric,
    )
    del df

    # constraints
    dict_monotone_constraints = {
        # better
        'fltgrossmonthly__income_sum': -1, # as income increases, prediction gets better
        'fltapproveddowntotal__app': -1,
        'fltdowncash__app': -1,
        'bookvalue__app': -1,
        'ENG-dealership_age': -1,
        'bookvalue__app': -1,
        'fltdowncash__app': -1,
        'fltapproveddowntotal__app': -1,
        # worse
        'fltgrossmonthly__income_count': 1, # as count of income increases, prediction gets worse
        'ENG-loan_to_value': 1,
        'ENG-payment_to_income': 1,
        'ENG-vehicle_age': 1,
        'fltadvance__app': 1,
        'bigmileage_odometer__app': 1,
        'amtfinanced__app': 1,
        'miles_odometer__app': 1,
        'pti__app': 1,
        'fltadvance__app': 1,
        'ENG-loan_to_value': 1,
        'amtfinanced__app': 1,
    }
    # ensure features are in list_cols_model
    dict_monotone_constraints = {key: val for key, val in dict_monotone_constraints.items() if key in list_cols_model}

    # init class
    cls_model_inference = cb.CatBoostClassifier(
        task_type='CPU',
        nan_mode='Min',
        random_state=42,
        eval_metric=str_eval_metric,
        iterations=int_n_iterations,
        learning_rate=flt_learning_rate,
        class_weights=None,
        monotone_constraints=dict_monotone_constraints,
    )

    # fit
    cls_model_inference.fit(
        pool_train,
        eval_set=[pool_valid],
        verbose=100,
        use_best_model=True,
        early_stopping_rounds=int(round(int_n_iterations*flt_prop_early_stopping)), 
    )
    del pool_train
    del pool_valid

    ################################################################################################
    # GET TRAINING EVAL METRIC
    ################################################################################################
    print('Getting training eval metric...')

    # import data
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/00_preprocessing/02_make_dfs/{str_filename_train}'
    df = pd.read_parquet(str_uri, columns=list_cols_import)

    # get predictions
    if str_eval_metric in ['AUC','PRAUC','Logloss']:
        # probabilities
        df['y_hat'] = cls_model_inference.predict_proba(df[cls_model_inference.feature_names_])[:,1]
    elif str_eval_metric in ['F1']:
        # class
        df['y_hat'] = cls_model_inference.predict(df[cls_model_inference.feature_names_])
    else:
        pass

    # save train predictions
    y_hat_train = list(df['y_hat'])

    # get eval metric - train
    if str_eval_metric == 'AUC':
        flt_eval_metric_train = skm.roc_auc_score(y_true=df[str_target], y_score=df['y_hat'])
    elif str_eval_metric == 'PRAUC':
        flt_eval_metric_train = skm.average_precision_score(y_true=df[str_target], y_score=df['y_hat'])
    elif str_eval_metric == 'Logloss':
        flt_eval_metric_train = skm.log_loss(y_true=df[str_target], y_pred=df['y_hat'])
    elif str_eval_metric == 'F1':
        flt_eval_metric_train = skm.f1_score(y_true=df[str_target], y_pred=df['y_hat'])

    # save memory
    del df

    ################################################################################################
    # GET VALIDATION EVAL METRIC (WEIGHTED)
    ################################################################################################
    print('Getting validation eval metric...')

    # import data
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/00_preprocessing/02_make_dfs/{str_filename_valid}'
    df = pd.read_parquet(str_uri, columns=list_cols_import)

    # get predictions
    if str_eval_metric in ['AUC','PRAUC','Logloss']:
        # probabilities
        df['y_hat'] = cls_model_inference.predict_proba(df[cls_model_inference.feature_names_])[:,1]
    elif str_eval_metric in ['F1']:
        # class
        df['y_hat'] = cls_model_inference.predict(df[cls_model_inference.feature_names_])
    else:
        pass

    # save valid predictions
    y_hat_valid = list(df['y_hat'])

    # get eval metric - train
    if str_eval_metric == 'AUC':
        flt_eval_metric_valid = skm.roc_auc_score(y_true=df[str_target], y_score=df['y_hat'])
    elif str_eval_metric == 'PRAUC':
        flt_eval_metric_valid = skm.average_precision_score(y_true=df[str_target], y_score=df['y_hat'])
    elif str_eval_metric == 'Logloss':
        flt_eval_metric_valid = skm.log_loss(y_true=df[str_target], y_pred=df['y_hat'])
    elif str_eval_metric == 'F1':
        flt_eval_metric_valid = skm.f1_score(y_true=df[str_target], y_pred=df['y_hat'])

    # save memory
    del df

    ################################################################################################
    # PICKLE MODEL AND PREDICTIONS
    ################################################################################################
    print('Pickling model and predictions...')

    flt_diff = abs(flt_eval_metric_train - flt_eval_metric_valid)
    dict_output = {
        'model_inference': cls_model_inference,
        'y_hat_train': y_hat_train,
        'flt_eval_metric_train': flt_eval_metric_train,
        'y_hat_valid': y_hat_valid,
        'flt_eval_metric_valid': flt_eval_metric_valid,
        'diff': flt_diff,
        'best_iteration': cls_model_inference.get_best_iteration(),
    }
    str_filename = f'dict_model_inference_{int_idx_array}.pkl'
    str_local_path = f'{str_dirname_output}/{str_filename}'
    pickle.dump(dict_output, open(str_local_path, 'wb'))

    ################################################################################################
    # UPLOAD TO S3
    ################################################################################################
    print('Uploading inference model to s3...')

    str_bucket_path = f'02_pricing_pd/02_model/02_model/02_batch_tuning/models/{str_filename}'
    upload_to_s3(
        str_local_path=str_local_path, 
        str_bucket_path=str_bucket_path, 
        str_project=str_project,
    )

    # create output data frame
    print('Creating output data frame...')
    dict_row = {
        'iteration': int_idx_array,
        'learning_rate': flt_learning_rate,
        'flt_eval_metric_train': flt_eval_metric_train,
        'flt_eval_metric_valid': flt_eval_metric_valid,
        'diff': flt_diff,
        'best_iteration': cls_model_inference.get_best_iteration(),
    }
    df = pd.DataFrame(dict_row, index=[0])
    # save
    str_filename = f'df_output_{int_idx_array}.csv'
    str_uri = f's3://{str_project}/02_pricing_pd/02_model/02_model/02_batch_tuning/models/{str_filename}'
    df.to_csv(str_uri, index=False)

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxii-pd-tuning-2

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  57.34kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
3.8: Pulling from lambda/python
2a445831cb62: Pulling fs layer
712150f0d27a: Pulling fs layer
a3d1d6f47e9c: Pulling fs layer
79a77e7c1be9: Pulling fs layer
687f68005109: Pulling fs layer
972510c161c2: Pulling fs layer
687f68005109: Waiting
972510c161c2: Waiting
a3d1d6f47e9c: Verifying Checksum
a3d1d6f47e9c: Download complete
712150f0d27a: Verifying Checksum
712150f0d27a: Download complete
79a77e7c1be9: Verifying Checksum
79a77e7c1be9: Download complete
972510c161c2: Verifying Checksum
972510c161c2: Download complete
687f68005109: Verifying Checksum
687f68005109: Download complete
2a445831cb62: Download complete
2a445831cb62: Pull complete
712150f0d27a: Pull complete
a3d1d6f47e9c: Pull complete
79a77e7c1be9: Pull complete
687f68005109: Pull complete
972510c161c2: Pull complete
Digest: sha256:d3541c8687af0fe6c25b09851626359ce83ae545cee1dfc40588f844e147a4c8
Status: Downloaded newer image for p

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 kB 11.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.6/153.6 kB 32.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.2/302.2 kB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.5/502.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.1/141.1 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.1/301.1 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-pd-tuning-2' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-pd-tuning-2]
6e97b5d07745: Preparing
215701132419: Preparing
6b56049841ae: Preparing
8b026f0465d7: Preparing
45d114d35c45: Preparing
86e4d644d316: Preparing
15dd6c63f3a2: Preparing
dd00ec5a5244: Preparing
8f43d000b361: Preparing
c349004e3af3: Preparing
86e4d644d316: Waiting
15dd6c63f3a2: Waiting
8f43d000b361: Waiting
dd00ec5a5244: Waiting
c349004e3af3: Waiting
6b56049841ae: Pushed
6e97b5d07745: Pushed
8b026f0465d7: Pushed
15dd6c63f3a2: Pushed
8f43d000b361: Pushed
dd00ec5a5244: Pushed
45d114d35c45: Pushed
86e4d644d316: Pushed
c349004e3af3: Pushed
215701132419: Pushed
latest: digest: sha256:65e2867ec1a6a421b61a05b1eb9d6b450e31a7585f9c574b06168f533b8c8113 size: 2420


### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

In [10]:
# create function
str_image_uri = f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_function_name}:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=900,
    MemorySize=1000,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 1000,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': '65e2867ec1a6a421b61a05b1eb9d6b450e31a7585f9c574b06168f533b8c8113',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 1000},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-pd-tuning-2',
 'FunctionName': 'genxii-pd-tuning-2',
 'LastModified': '2023-11-10T17:02:54.111+0000',
 'MemorySize': 1000,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1059',
                                      'content-type': 'application/json',
                                      'date': 'Fri, 10 Nov 2023 17:02:55 GMT',
                                      'x-amzn-requestid': '54ce616b-393c-4b5b-b9bf-7feda6e45893'},
                      'HTTPStatusCode': 201,
                      'RequestId': '54ce616b-393c-4b5b-b9bf-7feda6e45893',
                      'RetryAttempts': 0},
 'RevisionId': '46fc5eff-080c-4bcf-899c-3c671541dc

### Clean-up

In [11]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)